# DistilBERT + Retrieval-Augmented Inference

## 1. Setup & Imports

In [ ]:
!pip install -q faiss-gpu sentence-transformers
import os, re, random, warnings
os.environ["WANDB_MODE"] = "disabled"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from wordcloud import WordCloud
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import wandb
from sentence_transformers import SentenceTransformer
import faiss
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A','B','C','D','E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'

## 2. Data Loading & EDA

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

In [ ]:
plt.figure(figsize=(6,4))
train_df['answer'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Answer Distribution (Train)')
plt.xlabel('Option')
plt.ylabel('Count')
plt.show()

In [ ]:
def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)
train_df['option_set'] = train_df.apply(option_set_key, axis=1)
dup_count = train_df.duplicated(subset=['option_set']).sum()
print(f"Number of duplicate option-sets in train: {dup_count}")

In [ ]:
train_df['correct_len'] = train_df.apply(lambda row: len(str(row[row['answer']])), axis=1)
train_df['incorrect_lens'] = train_df.apply(
    lambda row: [len(str(row[l])) for l in LABELS if l != row['answer']], axis=1
)
incorrect_lens = train_df['incorrect_lens'].explode().reset_index(drop=True)
correct_lens = train_df['correct_len']

plot_data = pd.DataFrame({
    'length': pd.concat([correct_lens, incorrect_lens], ignore_index=True),
    'type': ['correct'] * len(correct_lens) + ['incorrect'] * len(incorrect_lens)
})

In [ ]:
plt.figure(figsize=(7, 5))

sns.barplot(
    x='type',
    y='length',
    data=plot_data,
    estimator='mean',
    errorbar=None,
    palette=['green', 'red']
)

plt.title('Average Option Length by Correctness')
plt.xlabel('Correctness')
plt.ylabel('Average Character Length')
plt.show()

In [ ]:
prompt_text = ' '.join(train_df['prompt_clean'].astype(str))
wordcloud = WordCloud(width=800, height=400, background_color='white',
                      max_words=100, random_state=SEED).generate(prompt_text)
plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Prompts')
plt.show()

In [ ]:
all_options = []
for _, row in train_df.iterrows():
    for l in LABELS:
        all_options.append(clean_text(row[l]))
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)
tfidf.fit(all_options)

sim_matrix = np.zeros((5,5))
for _, row in train_df.iterrows():
    opt_vecs = tfidf.transform([clean_text(row[l]) for l in LABELS])
    sim = cosine_similarity(opt_vecs)
    sim_matrix += sim
sim_matrix /= len(train_df)

plt.figure(figsize=(6,5))
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS)
plt.title('Average Cosine Similarity Between Options')
plt.show()

In [ ]:
prompt_cos_sims = {'correct': [], 'incorrect': []}
for _, row in train_df.iterrows():
    pv = tfidf.transform([clean_prompt(row['prompt'])])
    for l in LABELS:
        ov = tfidf.transform([clean_text(row[l])])
        sim = cosine_similarity(pv, ov)[0,0]
        if l == row['answer']:
            prompt_cos_sims['correct'].append(sim)
        else:
            prompt_cos_sims['incorrect'].append(sim)

plot_data2 = pd.DataFrame({
    'similarity': prompt_cos_sims['correct'] + prompt_cos_sims['incorrect'],
    'type': ['correct'] * len(prompt_cos_sims['correct']) +
            ['incorrect'] * len(prompt_cos_sims['incorrect'])
})

plt.figure(figsize=(7, 5))

sns.barplot(
    x='type',
    y='similarity',
    data=plot_data2,
    estimator='mean',
    errorbar=None,
    palette=['green', 'red']
)

plt.title('Average Cosine Similarity between Prompt and Option')
plt.xlabel('Correctness')
plt.ylabel('Average Cosine Similarity')
plt.ylim(0, 1)
plt.show()

## 2.1 Train–Test Overlap Analysis


In [ ]:
test_df['option_set'] = test_df.apply(option_set_key, axis=1)
train_option_sets = set(train_df['option_set'].unique())
test_matched = test_df['option_set'].isin(train_option_sets)

print(f"Unique option-sets in train: {len(train_option_sets)}")
print(f"Test questions matched to train: {test_matched.sum()} / {len(test_df)} ({test_matched.mean()*100:.1f}%)")
print(f"Test questions unmatched: {(~test_matched).sum()}")

plt.figure(figsize=(5,4))
pd.Series({'Matched': test_matched.sum(), 'Unmatched': (~test_matched).sum()}).plot(
    kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Train-Test Overlap (by Option Set)')
plt.ylabel('Count')
plt.show()

## 3. Data Preprocessing & Strict Splitting


In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[ra] = rb

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d:
            uf.union(i, d[k])
        else:
            d[k] = i
train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))
train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l:i for i,l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)
y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)} (zero overlap in option-sets)")

## 4. Retrieval Index Setup


In [ ]:
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))

def build_text(row):
    """Combine prompt + options into a single string for embedding."""
    p = clean_prompt(row['prompt'])
    opts = ' '.join(str(row[l]) for l in LABELS)
    return f"{p} {opts}"

train_texts = train_split.apply(build_text, axis=1).tolist()
test_texts  = test_df.apply(build_text, axis=1).tolist()
val_texts   = val_split.apply(build_text, axis=1).tolist()

print("Encoding training set with SBERT...")
train_embeds = sbert.encode(train_texts, show_progress_bar=True,
                            batch_size=64, normalize_embeddings=True)
test_embeds  = sbert.encode(test_texts,  show_progress_bar=True,
                            batch_size=64, normalize_embeddings=True)
val_embeds   = sbert.encode(val_texts,   show_progress_bar=True,
                            batch_size=64, normalize_embeddings=True)

